In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf
import random

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, average_precision_score
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization,
    Concatenate, Layer
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

# =====================================
# 0. 固定隨機種子
# =====================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# =====================================
# 1.5 反轉反向指標
# 分數越高 -> 越漂綠 / 風險越高
# =====================================
reverse_map = {
    "llama_vagueness_score_1": "llama_vagueness_risk_1",
    "llama_deflection_score_1": "llama_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中，無法建立反向指標")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)

# =====================================
# 2. target / groups
# =====================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")

y = df["label"].astype(int)
groups_all = df["Company"]

# =====================================
# 3. feature groups (LLaMA)
# =====================================
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_risk_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_risk_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# =====================================
# 4. 檢查欄位
# =====================================
all_needed_cols = semantic_cols + lexical_cols + financial_cols + ["label", "Company"]
missing_cols = [col for col in all_needed_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =====================================
# 5. Ablation sets
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# =====================================
# 6. outer CV
# =====================================
outer_cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)

# =====================================
# 7. 小 tuning grid（M1–M6 全跑）
# 這版只調：
# - learning_rate
# - batch_size
# - dropout_rate
# dense 結構保留你目前最常見且表現不錯的兩組
# =====================================
param_configs = [
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.2, "learning_rate": 1e-3, "batch_size": 16},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.2, "learning_rate": 1e-3, "batch_size": 32},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.2, "learning_rate": 5e-4, "batch_size": 16},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.2, "learning_rate": 5e-4, "batch_size": 32},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.2, "learning_rate": 1e-4, "batch_size": 16},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.2, "learning_rate": 1e-4, "batch_size": 32},

    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.3, "learning_rate": 1e-3, "batch_size": 16},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.3, "learning_rate": 1e-3, "batch_size": 32},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.3, "learning_rate": 5e-4, "batch_size": 16},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.3, "learning_rate": 5e-4, "batch_size": 32},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.3, "learning_rate": 1e-4, "batch_size": 16},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.3, "learning_rate": 1e-4, "batch_size": 32},

    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.2, "learning_rate": 1e-3, "batch_size": 16},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.2, "learning_rate": 1e-3, "batch_size": 32},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.2, "learning_rate": 5e-4, "batch_size": 16},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.2, "learning_rate": 5e-4, "batch_size": 32},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.2, "learning_rate": 1e-4, "batch_size": 16},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.2, "learning_rate": 1e-4, "batch_size": 32},

    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.3, "learning_rate": 1e-3, "batch_size": 16},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.3, "learning_rate": 1e-3, "batch_size": 32},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.3, "learning_rate": 5e-4, "batch_size": 16},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.3, "learning_rate": 5e-4, "batch_size": 32},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.3, "learning_rate": 1e-4, "batch_size": 16},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.3, "learning_rate": 1e-4, "batch_size": 32},
]

# =====================================
# 8. Feature Attention（sigmoid gating）
# =====================================
class FeatureAttention(Layer):
    def __init__(self, num_features, **kwargs):
        super().__init__(**kwargs)
        self.num_features = num_features
        self.score_dense = Dense(num_features)

    def call(self, inputs):
        scores = self.score_dense(inputs)
        weights = tf.nn.sigmoid(scores)
        attended = inputs * weights
        return attended

# =====================================
# 9. 建 MLP + Attention
# continuous: attention
# lexical: 小 Dense 後 concat
# =====================================
def build_mlp_attention(
    num_continuous,
    num_lexical,
    dense_1=64,
    dense_2=32,
    dropout_rate=0.2,
    learning_rate=1e-3,
    lexical_dense=16,
    l2_reg=1e-4
):
    cont_input = Input(shape=(num_continuous,), name="continuous_input")
    lex_input = Input(shape=(num_lexical,), name="lexical_input")

    if num_continuous > 0:
        cont_x = FeatureAttention(num_continuous, name="feature_attention")(cont_input)
        cont_x = BatchNormalization(name="bn_cont")(cont_x)
    else:
        cont_x = None

    if num_lexical > 0:
        lex_x = Dense(
            lexical_dense,
            activation="relu",
            kernel_regularizer=l2(l2_reg),
            name="lexical_dense"
        )(lex_input)
        lex_x = BatchNormalization(name="bn_lex")(lex_x)
        lex_x = Dropout(min(dropout_rate, 0.2), name="dropout_lex")(lex_x)
    else:
        lex_x = None

    if cont_x is not None and lex_x is not None:
        x = Concatenate(name="concat_cont_lex")([cont_x, lex_x])
    elif cont_x is not None:
        x = cont_x
    elif lex_x is not None:
        x = lex_x
    else:
        raise ValueError("num_continuous 與 num_lexical 不能同時為 0")

    x = Dense(
        dense_1,
        activation="relu",
        kernel_regularizer=l2(l2_reg),
        name="dense_1"
    )(x)
    x = BatchNormalization(name="bn_1")(x)
    x = Dropout(dropout_rate, name="dropout_1")(x)

    x = Dense(
        dense_2,
        activation="relu",
        kernel_regularizer=l2(l2_reg),
        name="dense_2"
    )(x)
    x = BatchNormalization(name="bn_2")(x)
    x = Dropout(dropout_rate, name="dropout_2")(x)

    outputs = Dense(1, activation="sigmoid", name="output")(x)

    model = Model(inputs=[cont_input, lex_input], outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# =====================================
# 10. 找最佳 threshold
# =====================================
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.10, 0.91, 0.01)
    best_threshold = 0.50
    best_score = -1

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = t

    return best_threshold, best_score

# =====================================
# 11. outer train 再切 validation（group-aware）
# =====================================
def make_group_validation_split(X_train_df, y_train_s, groups_train_s):
    inner_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    tr_idx, val_idx = next(inner_splitter.split(X_train_df, y_train_s, groups=groups_train_s))

    X_tr = X_train_df.iloc[tr_idx]
    X_val = X_train_df.iloc[val_idx]
    y_tr = y_train_s.iloc[tr_idx]
    y_val = y_train_s.iloc[val_idx]
    groups_tr = groups_train_s.iloc[tr_idx]
    groups_val = groups_train_s.iloc[val_idx]

    return X_tr, X_val, y_tr, y_val, groups_tr, groups_val

# =====================================
# 12. 自動拆 continuous / lexical
# =====================================
def split_feature_columns(selected_cols):
    cont_cols = [c for c in selected_cols if c in (semantic_cols + financial_cols)]
    lex_cols = [c for c in selected_cols if c in lexical_cols]
    return cont_cols, lex_cols

# =====================================
# 13. 單一 config 評估
# =====================================
def evaluate_single_config(df, y, groups, feature_name, selected_cols, config):
    fold_metrics = []
    best_thresholds = []

    cont_cols, lex_cols = split_feature_columns(selected_cols)

    print(f"\n=== {feature_name} | config={config} ===")
    print(f"Continuous cols: {cont_cols}")
    print(f"Lexical cols: {lex_cols}")

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(df[selected_cols], y, groups=groups), start=1
    ):
        print(f"[{feature_name}] Fold {fold_idx} started")

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()
        groups_train = groups.iloc[train_idx].copy()

        X_tr_df, X_val_df, y_tr, y_val, _, _ = make_group_validation_split(
            train_df[selected_cols], y_train, groups_train
        )

        # continuous
        if len(cont_cols) > 0:
            X_tr_cont_df = X_tr_df[cont_cols].copy()
            X_val_cont_df = X_val_df[cont_cols].copy()
            X_test_cont_df = test_df[cont_cols].copy()

            cont_imputer = SimpleImputer(strategy="median")
            X_tr_cont = cont_imputer.fit_transform(X_tr_cont_df)
            X_val_cont = cont_imputer.transform(X_val_cont_df)
            X_test_cont = cont_imputer.transform(X_test_cont_df)

            cont_scaler = StandardScaler()
            X_tr_cont = cont_scaler.fit_transform(X_tr_cont)
            X_val_cont = cont_scaler.transform(X_val_cont)
            X_test_cont = cont_scaler.transform(X_test_cont)

            X_tr_cont = X_tr_cont.astype(np.float32)
            X_val_cont = X_val_cont.astype(np.float32)
            X_test_cont = X_test_cont.astype(np.float32)
        else:
            X_tr_cont = np.zeros((len(X_tr_df), 0), dtype=np.float32)
            X_val_cont = np.zeros((len(X_val_df), 0), dtype=np.float32)
            X_test_cont = np.zeros((len(test_df), 0), dtype=np.float32)

        # lexical
        if len(lex_cols) > 0:
            X_tr_lex_df = X_tr_df[lex_cols].copy()
            X_val_lex_df = X_val_df[lex_cols].copy()
            X_test_lex_df = test_df[lex_cols].copy()

            lex_imputer = SimpleImputer(strategy="most_frequent")
            X_tr_lex = lex_imputer.fit_transform(X_tr_lex_df)
            X_val_lex = lex_imputer.transform(X_val_lex_df)
            X_test_lex = lex_imputer.transform(X_test_lex_df)

            X_tr_lex = X_tr_lex.astype(np.float32)
            X_val_lex = X_val_lex.astype(np.float32)
            X_test_lex = X_test_lex.astype(np.float32)
        else:
            X_tr_lex = np.zeros((len(X_tr_df), 0), dtype=np.float32)
            X_val_lex = np.zeros((len(X_val_df), 0), dtype=np.float32)
            X_test_lex = np.zeros((len(test_df), 0), dtype=np.float32)

        classes = np.unique(y_tr)
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=y_tr
        )
        class_weight_dict = {
            int(cls): float(w) for cls, w in zip(classes, class_weights)
        }

        tf.keras.backend.clear_session()
        model = build_mlp_attention(
            num_continuous=X_tr_cont.shape[1],
            num_lexical=X_tr_lex.shape[1],
            dense_1=config["dense_1"],
            dense_2=config["dense_2"],
            dropout_rate=config["dropout_rate"],
            learning_rate=config["learning_rate"]
        )

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=10,
            min_delta=1e-4,
            restore_best_weights=True
        )

        model.fit(
            [X_tr_cont, X_tr_lex], y_tr,
            validation_data=([X_val_cont, X_val_lex], y_val),
            epochs=100,
            batch_size=config["batch_size"],
            class_weight=class_weight_dict,
            callbacks=[early_stop],
            verbose=0
        )

        val_prob = model.predict([X_val_cont, X_val_lex], verbose=0).ravel()
        best_threshold, _ = find_best_threshold(y_val, val_prob)
        best_thresholds.append(best_threshold)

        test_prob = model.predict([X_test_cont, X_test_lex], verbose=0).ravel()
        y_pred = (test_prob >= best_threshold).astype(int)

        try:
            roc_auc = roc_auc_score(y_test, test_prob)
        except ValueError:
            roc_auc = np.nan

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        print(
            f"[{feature_name}] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f}"
        )

    return {
        "Model": "MLP + Attention (small tuning full run)",
        "Feature_Set": feature_name,
        "Config": str(config),
        "Num_Features": len(selected_cols),
        "Num_Continuous": len(cont_cols),
        "Num_Lexical": len(lex_cols),

        "Accuracy_mean": np.nanmean([m["accuracy"] for m in fold_metrics]),
        "F1_mean": np.nanmean([m["f1"] for m in fold_metrics]),
        "ROC_AUC_mean": np.nanmean([m["roc_auc"] for m in fold_metrics]),
        "Precision_mean": np.nanmean([m["precision"] for m in fold_metrics]),
        "Recall_mean": np.nanmean([m["recall"] for m in fold_metrics]),
        "PR_AUC_mean": np.nanmean([m["average_precision"] for m in fold_metrics]),

        "Accuracy_std": np.nanstd([m["accuracy"] for m in fold_metrics], ddof=1),
        "F1_std": np.nanstd([m["f1"] for m in fold_metrics], ddof=1),
        "ROC_AUC_std": np.nanstd([m["roc_auc"] for m in fold_metrics], ddof=1),
        "Precision_std": np.nanstd([m["precision"] for m in fold_metrics], ddof=1),
        "Recall_std": np.nanstd([m["recall"] for m in fold_metrics], ddof=1),
        "PR_AUC_std": np.nanstd([m["average_precision"] for m in fold_metrics], ddof=1),

        "Mean_Best_Threshold": np.nanmean(best_thresholds),
        "Threshold_std": np.nanstd(best_thresholds, ddof=1)
    }

# =====================================
# 14. 每個 feature set 跑全部 configs
# =====================================
all_results = []

for feature_name, cols in feature_sets.items():
    for config in param_configs:
        result = evaluate_single_config(
            df=df,
            y=y,
            groups=groups_all,
            feature_name=feature_name,
            selected_cols=cols,
            config=config
        )
        all_results.append(result)

# =====================================
# 15. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\n===== All Results =====")
print(results_df)

results_df.to_csv(
    "llama_MLP_attention_small_tuning_full_M1_M6.csv",
    index=False,
    encoding="utf-8-sig"
)

# =====================================
# 16. 每個 feature set 挑 F1 最佳 config
# =====================================
best_results_df = (
    results_df.sort_values(["Feature_Set", "F1_mean"], ascending=[True, False])
    .groupby("Feature_Set", as_index=False)
    .first()
)

print("\n===== Best Config per Feature Set =====")
print(best_results_df)

best_results_df.to_csv(
    "llama_MLP_attention_small_tuning_best_per_featureset_M1_M6.csv",
    index=False,
    encoding="utf-8-sig"
)

# =====================================
# 17. 檢查反轉是否成功（可選）
# =====================================
print("\n===== Check reversed indicators =====")
print(df[[
    "llama_vagueness_score_1", "llama_vagueness_risk_1",
    "llama_deflection_score_1", "llama_deflection_risk_1"
]].head())


=== M1: Semantic | config={'dense_1': 64, 'dense_2': 32, 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 16} ===
Continuous cols: ['llama_specificity_score_1', 'llama_evidence_substantiation_score_1', 'llama_vagueness_risk_1', 'llama_commitment_score_1', 'llama_temporal_credibility_score_1', 'llama_deflection_risk_1', 'llama_comparability_score_1']
Lexical cols: []
[M1: Semantic] Fold 1 started
[M1: Semantic] Fold 1 done | Threshold=0.46 | F1=0.6000
[M1: Semantic] Fold 2 started
[M1: Semantic] Fold 2 done | Threshold=0.65 | F1=1.0000
[M1: Semantic] Fold 3 started
[M1: Semantic] Fold 3 done | Threshold=0.81 | F1=0.4000
[M1: Semantic] Fold 4 started
[M1: Semantic] Fold 4 done | Threshold=0.81 | F1=0.0000
[M1: Semantic] Fold 5 started
[M1: Semantic] Fold 5 done | Threshold=0.21 | F1=0.8000
[M1: Semantic] Fold 6 started
[M1: Semantic] Fold 6 done | Threshold=0.72 | F1=0.7273
[M1: Semantic] Fold 7 started
[M1: Semantic] Fold 7 done | Threshold=0.42 | F1=0.2857
[M1: Semantic] Fol

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import random

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, average_precision_score
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization,
    Concatenate, Layer
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

# =====================================
# 0. 固定隨機種子
# =====================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# =====================================
# 1.5 反轉反向指標
# 分數越高 -> 越漂綠 / 風險越高
# =====================================
reverse_map = {
    "chatgpt_vagueness_score_1": "chatgpt_vagueness_risk_1",
    "chatgpt_deflection_score_1": "chatgpt_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中，無法建立反向指標")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)

# =====================================
# 2. target / groups
# =====================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")

y = df["label"].astype(int)
groups_all = df["Company"]

# =====================================
# 3. feature groups (LLaMA)
# =====================================
semantic_cols = [
    "chatgpt_specificity_score_1",
    "chatgpt_evidence_substantiation_score_1",
    "chatgpt_vagueness_risk_1",
    "chatgpt_commitment_score_1",
    "chatgpt_temporal_credibility_score_1",
    "chatgpt_deflection_risk_1",
    "chatgpt_comparability_score_1"
]

lexical_cols = [
    "chatgpt_has_scope_1",
    "chatgpt_has_sbti_1",
    "chatgpt_has_material_1",
    "chatgpt_has_kpi_1",
    "chatgpt_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# =====================================
# 4. 檢查欄位
# =====================================
all_needed_cols = semantic_cols + lexical_cols + financial_cols + ["label", "Company"]
missing_cols = [col for col in all_needed_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =====================================
# 5. Ablation sets
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# =====================================
# 6. outer CV
# =====================================
outer_cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)

# =====================================
# 7. 小 tuning grid（M1–M6 全跑）
# 這版只調：
# - learning_rate
# - batch_size
# - dropout_rate
# dense 結構保留你目前最常見且表現不錯的兩組
# =====================================
param_configs = [
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.2, "learning_rate": 1e-3, "batch_size": 16},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.2, "learning_rate": 1e-3, "batch_size": 32},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.2, "learning_rate": 5e-4, "batch_size": 16},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.2, "learning_rate": 5e-4, "batch_size": 32},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.2, "learning_rate": 1e-4, "batch_size": 16},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.2, "learning_rate": 1e-4, "batch_size": 32},

    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.3, "learning_rate": 1e-3, "batch_size": 16},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.3, "learning_rate": 1e-3, "batch_size": 32},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.3, "learning_rate": 5e-4, "batch_size": 16},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.3, "learning_rate": 5e-4, "batch_size": 32},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.3, "learning_rate": 1e-4, "batch_size": 16},
    {"dense_1": 64,  "dense_2": 32, "dropout_rate": 0.3, "learning_rate": 1e-4, "batch_size": 32},

    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.2, "learning_rate": 1e-3, "batch_size": 16},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.2, "learning_rate": 1e-3, "batch_size": 32},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.2, "learning_rate": 5e-4, "batch_size": 16},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.2, "learning_rate": 5e-4, "batch_size": 32},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.2, "learning_rate": 1e-4, "batch_size": 16},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.2, "learning_rate": 1e-4, "batch_size": 32},

    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.3, "learning_rate": 1e-3, "batch_size": 16},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.3, "learning_rate": 1e-3, "batch_size": 32},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.3, "learning_rate": 5e-4, "batch_size": 16},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.3, "learning_rate": 5e-4, "batch_size": 32},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.3, "learning_rate": 1e-4, "batch_size": 16},
    {"dense_1": 128, "dense_2": 64, "dropout_rate": 0.3, "learning_rate": 1e-4, "batch_size": 32},
]

# =====================================
# 8. Feature Attention（sigmoid gating）
# =====================================
class FeatureAttention(Layer):
    def __init__(self, num_features, **kwargs):
        super().__init__(**kwargs)
        self.num_features = num_features
        self.score_dense = Dense(num_features)

    def call(self, inputs):
        scores = self.score_dense(inputs)
        weights = tf.nn.sigmoid(scores)
        attended = inputs * weights
        return attended

# =====================================
# 9. 建 MLP + Attention
# continuous: attention
# lexical: 小 Dense 後 concat
# =====================================
def build_mlp_attention(
    num_continuous,
    num_lexical,
    dense_1=64,
    dense_2=32,
    dropout_rate=0.2,
    learning_rate=1e-3,
    lexical_dense=16,
    l2_reg=1e-4
):
    cont_input = Input(shape=(num_continuous,), name="continuous_input")
    lex_input = Input(shape=(num_lexical,), name="lexical_input")

    if num_continuous > 0:
        cont_x = FeatureAttention(num_continuous, name="feature_attention")(cont_input)
        cont_x = BatchNormalization(name="bn_cont")(cont_x)
    else:
        cont_x = None

    if num_lexical > 0:
        lex_x = Dense(
            lexical_dense,
            activation="relu",
            kernel_regularizer=l2(l2_reg),
            name="lexical_dense"
        )(lex_input)
        lex_x = BatchNormalization(name="bn_lex")(lex_x)
        lex_x = Dropout(min(dropout_rate, 0.2), name="dropout_lex")(lex_x)
    else:
        lex_x = None

    if cont_x is not None and lex_x is not None:
        x = Concatenate(name="concat_cont_lex")([cont_x, lex_x])
    elif cont_x is not None:
        x = cont_x
    elif lex_x is not None:
        x = lex_x
    else:
        raise ValueError("num_continuous 與 num_lexical 不能同時為 0")

    x = Dense(
        dense_1,
        activation="relu",
        kernel_regularizer=l2(l2_reg),
        name="dense_1"
    )(x)
    x = BatchNormalization(name="bn_1")(x)
    x = Dropout(dropout_rate, name="dropout_1")(x)

    x = Dense(
        dense_2,
        activation="relu",
        kernel_regularizer=l2(l2_reg),
        name="dense_2"
    )(x)
    x = BatchNormalization(name="bn_2")(x)
    x = Dropout(dropout_rate, name="dropout_2")(x)

    outputs = Dense(1, activation="sigmoid", name="output")(x)

    model = Model(inputs=[cont_input, lex_input], outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# =====================================
# 10. 找最佳 threshold
# =====================================
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.10, 0.91, 0.01)
    best_threshold = 0.50
    best_score = -1

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = t

    return best_threshold, best_score

# =====================================
# 11. outer train 再切 validation（group-aware）
# =====================================
def make_group_validation_split(X_train_df, y_train_s, groups_train_s):
    inner_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    tr_idx, val_idx = next(inner_splitter.split(X_train_df, y_train_s, groups=groups_train_s))

    X_tr = X_train_df.iloc[tr_idx]
    X_val = X_train_df.iloc[val_idx]
    y_tr = y_train_s.iloc[tr_idx]
    y_val = y_train_s.iloc[val_idx]
    groups_tr = groups_train_s.iloc[tr_idx]
    groups_val = groups_train_s.iloc[val_idx]

    return X_tr, X_val, y_tr, y_val, groups_tr, groups_val

# =====================================
# 12. 自動拆 continuous / lexical
# =====================================
def split_feature_columns(selected_cols):
    cont_cols = [c for c in selected_cols if c in (semantic_cols + financial_cols)]
    lex_cols = [c for c in selected_cols if c in lexical_cols]
    return cont_cols, lex_cols

# =====================================
# 13. 單一 config 評估
# =====================================
def evaluate_single_config(df, y, groups, feature_name, selected_cols, config):
    fold_metrics = []
    best_thresholds = []

    cont_cols, lex_cols = split_feature_columns(selected_cols)

    print(f"\n=== {feature_name} | config={config} ===")
    print(f"Continuous cols: {cont_cols}")
    print(f"Lexical cols: {lex_cols}")

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(df[selected_cols], y, groups=groups), start=1
    ):
        print(f"[{feature_name}] Fold {fold_idx} started")

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()
        groups_train = groups.iloc[train_idx].copy()

        X_tr_df, X_val_df, y_tr, y_val, _, _ = make_group_validation_split(
            train_df[selected_cols], y_train, groups_train
        )

        # continuous
        if len(cont_cols) > 0:
            X_tr_cont_df = X_tr_df[cont_cols].copy()
            X_val_cont_df = X_val_df[cont_cols].copy()
            X_test_cont_df = test_df[cont_cols].copy()

            cont_imputer = SimpleImputer(strategy="median")
            X_tr_cont = cont_imputer.fit_transform(X_tr_cont_df)
            X_val_cont = cont_imputer.transform(X_val_cont_df)
            X_test_cont = cont_imputer.transform(X_test_cont_df)

            cont_scaler = StandardScaler()
            X_tr_cont = cont_scaler.fit_transform(X_tr_cont)
            X_val_cont = cont_scaler.transform(X_val_cont)
            X_test_cont = cont_scaler.transform(X_test_cont)

            X_tr_cont = X_tr_cont.astype(np.float32)
            X_val_cont = X_val_cont.astype(np.float32)
            X_test_cont = X_test_cont.astype(np.float32)
        else:
            X_tr_cont = np.zeros((len(X_tr_df), 0), dtype=np.float32)
            X_val_cont = np.zeros((len(X_val_df), 0), dtype=np.float32)
            X_test_cont = np.zeros((len(test_df), 0), dtype=np.float32)

        # lexical
        if len(lex_cols) > 0:
            X_tr_lex_df = X_tr_df[lex_cols].copy()
            X_val_lex_df = X_val_df[lex_cols].copy()
            X_test_lex_df = test_df[lex_cols].copy()

            lex_imputer = SimpleImputer(strategy="most_frequent")
            X_tr_lex = lex_imputer.fit_transform(X_tr_lex_df)
            X_val_lex = lex_imputer.transform(X_val_lex_df)
            X_test_lex = lex_imputer.transform(X_test_lex_df)

            X_tr_lex = X_tr_lex.astype(np.float32)
            X_val_lex = X_val_lex.astype(np.float32)
            X_test_lex = X_test_lex.astype(np.float32)
        else:
            X_tr_lex = np.zeros((len(X_tr_df), 0), dtype=np.float32)
            X_val_lex = np.zeros((len(X_val_df), 0), dtype=np.float32)
            X_test_lex = np.zeros((len(test_df), 0), dtype=np.float32)

        classes = np.unique(y_tr)
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=y_tr
        )
        class_weight_dict = {
            int(cls): float(w) for cls, w in zip(classes, class_weights)
        }

        tf.keras.backend.clear_session()
        model = build_mlp_attention(
            num_continuous=X_tr_cont.shape[1],
            num_lexical=X_tr_lex.shape[1],
            dense_1=config["dense_1"],
            dense_2=config["dense_2"],
            dropout_rate=config["dropout_rate"],
            learning_rate=config["learning_rate"]
        )

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=10,
            min_delta=1e-4,
            restore_best_weights=True
        )

        model.fit(
            [X_tr_cont, X_tr_lex], y_tr,
            validation_data=([X_val_cont, X_val_lex], y_val),
            epochs=100,
            batch_size=config["batch_size"],
            class_weight=class_weight_dict,
            callbacks=[early_stop],
            verbose=0
        )

        val_prob = model.predict([X_val_cont, X_val_lex], verbose=0).ravel()
        best_threshold, _ = find_best_threshold(y_val, val_prob)
        best_thresholds.append(best_threshold)

        test_prob = model.predict([X_test_cont, X_test_lex], verbose=0).ravel()
        y_pred = (test_prob >= best_threshold).astype(int)

        try:
            roc_auc = roc_auc_score(y_test, test_prob)
        except ValueError:
            roc_auc = np.nan

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        print(
            f"[{feature_name}] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f}"
        )

    return {
        "Model": "MLP + Attention (small tuning full run)",
        "Feature_Set": feature_name,
        "Config": str(config),
        "Num_Features": len(selected_cols),
        "Num_Continuous": len(cont_cols),
        "Num_Lexical": len(lex_cols),

        "Accuracy_mean": np.nanmean([m["accuracy"] for m in fold_metrics]),
        "F1_mean": np.nanmean([m["f1"] for m in fold_metrics]),
        "ROC_AUC_mean": np.nanmean([m["roc_auc"] for m in fold_metrics]),
        "Precision_mean": np.nanmean([m["precision"] for m in fold_metrics]),
        "Recall_mean": np.nanmean([m["recall"] for m in fold_metrics]),
        "PR_AUC_mean": np.nanmean([m["average_precision"] for m in fold_metrics]),

        "Accuracy_std": np.nanstd([m["accuracy"] for m in fold_metrics], ddof=1),
        "F1_std": np.nanstd([m["f1"] for m in fold_metrics], ddof=1),
        "ROC_AUC_std": np.nanstd([m["roc_auc"] for m in fold_metrics], ddof=1),
        "Precision_std": np.nanstd([m["precision"] for m in fold_metrics], ddof=1),
        "Recall_std": np.nanstd([m["recall"] for m in fold_metrics], ddof=1),
        "PR_AUC_std": np.nanstd([m["average_precision"] for m in fold_metrics], ddof=1),

        "Mean_Best_Threshold": np.nanmean(best_thresholds),
        "Threshold_std": np.nanstd(best_thresholds, ddof=1)
    }

# =====================================
# 14. 每個 feature set 跑全部 configs
# =====================================
all_results = []

for feature_name, cols in feature_sets.items():
    for config in param_configs:
        result = evaluate_single_config(
            df=df,
            y=y,
            groups=groups_all,
            feature_name=feature_name,
            selected_cols=cols,
            config=config
        )
        all_results.append(result)

# =====================================
# 15. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\n===== All Results =====")
print(results_df)

results_df.to_csv(
    "chatgpt_MLP_attention_small_tuning_full_M1_M6.csv",
    index=False,
    encoding="utf-8-sig"
)

# =====================================
# 16. 每個 feature set 挑 F1 最佳 config
# =====================================
best_results_df = (
    results_df.sort_values(["Feature_Set", "F1_mean"], ascending=[True, False])
    .groupby("Feature_Set", as_index=False)
    .first()
)

print("\n===== Best Config per Feature Set =====")
print(best_results_df)

best_results_df.to_csv(
    "chatgpt_MLP_attention_small_tuning_best_per_featureset_M1_M6.csv",
    index=False,
    encoding="utf-8-sig"
)

# =====================================
# 17. 檢查反轉是否成功（可選）
# =====================================
print("\n===== Check reversed indicators =====")
print(df[[
    "chatgpt_vagueness_score_1", "chatgpt_vagueness_risk_1",
    "chatgpt_deflection_score_1", "chatgpt_deflection_risk_1"
]].head())


=== M1: Semantic | config={'dense_1': 64, 'dense_2': 32, 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 16} ===
Continuous cols: ['chatgpt_specificity_score_1', 'chatgpt_evidence_substantiation_score_1', 'chatgpt_vagueness_risk_1', 'chatgpt_commitment_score_1', 'chatgpt_temporal_credibility_score_1', 'chatgpt_deflection_risk_1', 'chatgpt_comparability_score_1']
Lexical cols: []
[M1: Semantic] Fold 1 started
[M1: Semantic] Fold 1 done | Threshold=0.36 | F1=1.0000
[M1: Semantic] Fold 2 started
[M1: Semantic] Fold 2 done | Threshold=0.63 | F1=0.6667
[M1: Semantic] Fold 3 started
[M1: Semantic] Fold 3 done | Threshold=0.84 | F1=0.8571
[M1: Semantic] Fold 4 started
[M1: Semantic] Fold 4 done | Threshold=0.90 | F1=0.2500
[M1: Semantic] Fold 5 started
[M1: Semantic] Fold 5 done | Threshold=0.89 | F1=0.9091
[M1: Semantic] Fold 6 started
[M1: Semantic] Fold 6 done | Threshold=0.54 | F1=0.9091
[M1: Semantic] Fold 7 started
[M1: Semantic] Fold 7 done | Threshold=0.77 | F1=0.5000
[M1: